# Prediction Reconciliation — Target Week 2026-05-11

Compares predicted values for `START_OF_WEEK = 2026-05-11` across sources that share the
same forward-looking forecast-horizon (`forecast_step` = weeks-ahead) semantics:

| # | Source | Dataset / file | Notes |
|---|--------|---------------|-------|
| 1 | **Cache** | `forecast_cache.csv` | Built by `prepare_forecast_cache.py` |
| 2 | **Live re-run** | DR deployment API | Fresh `run_predictions()` per FD |
| 3 | **DR predictions export** | `ts_predictions.csv` (`6a4b844b11847521f9a71d56`) | Monitoring Data Export — predictions only |

`ts_historical_prediction_data.csv` (`6a4b844a11847521f9a71d4a`) is **not** used for the
value comparison above — its `FORECAST_DISTANCE` is a backtest offset (weeks *before* the
export's snapshot date), not a forward forecast horizon, and it stores one row per historical
week rather than one row per FD. Instead it's used later (§6) to compare the **feature inputs**
DR's own batch job actually scored against, versus what our `PLANNED`/`ACTUAL` datasets
reconstruct for the same week — since live/cache calls and the historical batch job can draw
on different feature snapshots.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../").resolve()
sys.path.insert(0, str(REPO_ROOT))

import os
import pandas as pd
import datarobot as dr

# ── Load .env (Codespace path, falls back silently if not found) ──────────────
try:
    from dotenv import load_dotenv
    _env_candidates = [
        Path("/home/notebooks/storage/forecast-assistant/.env"),  # DR Codespace
        REPO_ROOT / ".env",                                        # local dev
    ]
    for _env_path in _env_candidates:
        if _env_path.exists():
            load_dotenv(_env_path, override=False)
            print(f".env loaded: {_env_path}")
            break
    else:
        print("No .env file found — relying on environment variables already set")
except ImportError:
    print("python-dotenv not installed — relying on environment variables already set")

# ── Config ────────────────────────────────────────────────────────────────────
TARGET_WEEK = "2026-05-11"

# Cache file location — candidates in priority order:
#   1. notebooks/data/  (DR Codespace: manually copied here)
#   2. data/            (repo root, local dev / deployed app)
_cache_candidates = [
    REPO_ROOT / "notebooks" / "data" / "forecast_cache.csv",
    REPO_ROOT / "data" / "forecast_cache.csv",
]
CACHE_FILE = next((p for p in _cache_candidates if p.exists()), _cache_candidates[0])

# DR AI Catalog — Data Export datasets (from Deployment → Monitoring → Data Exploration)
DR_PREDICTIONS_DATASET_ID     = "6a4b844b11847521f9a71d56"  # ts_predictions.csv (5876 rows, 5 cols)
DR_HISTORICAL_PRED_DATASET_ID = "6a4b844a11847521f9a71d4a"  # ts_historical_prediction_data.csv (130 rows, 43 cols)

# Feature datasets
PLANNED_DATASET_ID = os.environ.get("PLANNED_DATASET_ID", "6a326e0a76da3420b0d4e6e1")
ACTUAL_DATASET_ID  = os.environ.get("ACTUAL_DATASET_ID",  "6a326f4d347b28ea2e55f572")

# ── App settings overrides ────────────────────────────────────────────────────
os.environ.setdefault("FORECAST_DEPLOYMENT_ID", "6a0ec47bf306847615758fa3")

print("Repo root:", REPO_ROOT)
print("Cache:", CACHE_FILE, "| exists:", CACHE_FILE.exists())
print("FORECAST_DEPLOYMENT_ID     :", os.environ.get("FORECAST_DEPLOYMENT_ID", "NOT SET"))
print("FORECAST_SCORING_DATASET_ID:", os.environ.get("FORECAST_SCORING_DATASET_ID", "NOT SET"))

## 1 — Cache values for 2026-05-11

In [ ]:
cache = pd.read_csv(CACHE_FILE)
pred_col = next(c for c in cache.columns if "PREDICTION" in c)
print(f"Prediction column: {pred_col}")

cache_target = (
    cache[cache["START_OF_WEEK"].astype(str).str[:10] == TARGET_WEEK]
    [["prediction_week", "scenario", "forecast_step", pred_col]]
    .sort_values(["forecast_step", "scenario"])
    .reset_index(drop=True)
)
print(f"Cache rows for {TARGET_WEEK}: {len(cache_target)}")
cache_target

### Reference: Snowflake export (from screenshot)

| FC distance | Value |
|-------------|-------|
| 1 | 44,643 |
| 2 | 46,643 |
| 4 | 45,756 |

## 2 — Live re-run via `run_predictions()`

In [ ]:
from forecastic.comparison_api import _build_scoring_input_for_week, run_predictions
from forecastic.api import get_app_settings, scoring_dataset_id

app_settings = get_app_settings()
date_col   = app_settings.datetime_partition_column
target_col = app_settings.target
fw_end_wks = app_settings.forecast_window_end
print("date_col:", date_col, "| target_col:", target_col, "| fw_end:", fw_end_wks)

In [ ]:
# Load feature source datasets
DATA_DIR     = REPO_ROOT / "data"
PLANNED_FILE = next(DATA_DIR.glob("*PLANNED*"), None)
ACTUAL_FILE  = next(DATA_DIR.glob("*ACTUAL*FEATURES*"), None) or next(DATA_DIR.glob("*ACTUAL*"), None)

if PLANNED_FILE and ACTUAL_FILE:
    planned_src = pd.read_csv(PLANNED_FILE, parse_dates=[date_col])
    actual_src  = pd.read_csv(ACTUAL_FILE,  parse_dates=[date_col])
    print("Feature data: local CSV")
else:
    planned_src = dr.Dataset.get(PLANNED_DATASET_ID).get_as_dataframe()
    actual_src  = dr.Dataset.get(ACTUAL_DATASET_ID).get_as_dataframe()
    print("Feature data: DR AI Catalog")

scoring_src = dr.Dataset.get(scoring_dataset_id).get_as_dataframe()
scoring_src[date_col] = pd.to_datetime(scoring_src[date_col]).dt.strftime("%Y-%m-%d")
print(f"Scoring: {len(scoring_src)} rows  {scoring_src[date_col].min()} → {scoring_src[date_col].max()}")
print(f"Planned: {len(planned_src)} rows | Actual: {len(actual_src)} rows")

In [ ]:
from pandas import Timestamp, Timedelta

target_dt  = Timestamp(TARGET_WEEK)
pred_weeks = [
    (target_dt - Timedelta(weeks=fd)).strftime("%Y-%m-%d")
    for fd in range(1, fw_end_wks + 1)
]
print("Prediction weeks to test:", pred_weeks)

In [ ]:
live_results = []

for pred_week in pred_weeks:
    for scenario, fw_df in [("planned", planned_src), ("actual", actual_src)]:
        scoring_input = _build_scoring_input_for_week(
            scoring_df=scoring_src,
            prediction_week=pred_week,
            fw_features_df=fw_df,
        )
        if scoring_input.empty:
            print(f"  [{pred_week}][{scenario}] SKIP — no FDW rows")
            continue
        # Drop ASSOCIATION_ID so the deployment regenerates it uniformly.
        # FW rows from planned_src lack this column; after concat with FDW rows
        # that have it, planned FW rows get NaN → 422 from DR.
        scoring_input = scoring_input.drop(columns=["ASSOCIATION_ID"], errors="ignore")
        try:
            preds = run_predictions(scoring_input)
        except Exception as e:
            print(f"  [{pred_week}][{scenario}] ERROR: {e}")
            continue
        target_rows = [r for r in preds if str(r.get(date_col, ""))[:10] == TARGET_WEEK]
        for r in target_rows:
            live_results.append({
                "prediction_week": pred_week,
                "scenario": scenario,
                "forecast_step": r.get("forecast_step"),
                f"{pred_col}_LIVE": round(r.get(pred_col, float("nan")), 3),
            })
        print(f"  [{pred_week}][{scenario}] OK — {len(target_rows)} row(s)")

live_df = pd.DataFrame(live_results).sort_values(["forecast_step", "scenario"]).reset_index(drop=True)
live_df

## 3 — Cache vs Live comparison

**Expected:** `planned` diff ≈ 0 (planned feature dataset is static — same data at cache build time and now).

**Non-zero `actual` diff** means the actual features dataset in AI Catalog was **updated after the cache was built** — newer observed values (weather actuals, demand counts, etc.) replaced what was there when `prepare_forecast_cache.py` ran. This is expected behaviour: actuals arrive retrospectively.

**NaN in `_CACHE`** means that prediction week was not included in the cache build range.

In [ ]:
cache_r = cache_target.rename(columns={pred_col: f"{pred_col}_CACHE"})

# Use right join: live_df drives the rows (covers all FDs regardless of cache coverage).
# Cache values fill in where the prediction_week/scenario/forecast_step key exists in cache;
# NaN in _CACHE means that week was never cached (cache was built from a later start date).
comparison = live_df[["prediction_week", "scenario", "forecast_step", f"{pred_col}_LIVE"]].merge(
    cache_r[["prediction_week", "scenario", "forecast_step", f"{pred_col}_CACHE"]],
    on=["prediction_week", "scenario", "forecast_step"],
    how="left",
)

comparison["diff"]   = (comparison[f"{pred_col}_LIVE"] - comparison[f"{pred_col}_CACHE"]).round(3)
comparison["diff_%"] = (comparison["diff"] / comparison[f"{pred_col}_CACHE"] * 100).round(2)
comparison.sort_values(["forecast_step", "scenario"]).reset_index(drop=True)

## 4 — DR Data Export datasets

### 4a — `ts_predictions.csv` (5,876 rows, 5 cols)
Compact predictions-only export. Contains one row per ASSOCIATION_ID × prediction request.
Covers 2026-05-03 → 2026-09-21.

In [ ]:
print(f"Loading ts_predictions from catalog: {DR_PREDICTIONS_DATASET_ID}")
ts_preds = dr.Dataset.get(DR_PREDICTIONS_DATASET_ID).get_as_dataframe()
print("Shape:", ts_preds.shape)
print("Columns:", list(ts_preds.columns))
ts_preds.head(3)

In [ ]:
# Identify the date columns — adjust names if they differ from below
ts_preds.dtypes

In [ ]:
# ── Filter to TARGET_WEEK ─────────────────────────────────────────────────────
# Identify date-like columns automatically
date_like = [c for c in ts_preds.columns if ts_preds[c].dtype == object
             and ts_preds[c].str.match(r"\d{4}-\d{2}-\d{2}", na=False).any()]
print("Date-like columns:", date_like)

# Typical column names from DR export — adjust if needed
TS_TARGET_COL = "START_OF_WEEK"   # or the column containing the forecast target date
TS_PRED_COL   = "DR_RESERVED_PREDICTION_VALUE" if "DR_RESERVED_PREDICTION_VALUE" in ts_preds.columns else pred_col
TS_POINT_COL  = "DR_RESERVED_PREDICTION_TIMESTAMP" if "DR_RESERVED_PREDICTION_TIMESTAMP" in ts_preds.columns else "FORECAST_POINT"
TS_FD_COL     = "forecast_distance"  # export's own forecast-horizon column — NOT derived from TS_POINT_COL

# Fall back to first date-like col if exact names don't exist
if TS_TARGET_COL not in ts_preds.columns and date_like:
    TS_TARGET_COL = date_like[0]
    print(f"  → using '{TS_TARGET_COL}' as target date column")

ts_target = ts_preds[ts_preds[TS_TARGET_COL].astype(str).str[:10] == TARGET_WEEK].copy()
print(f"\nRows for {TARGET_WEEK}: {len(ts_target)}")
ts_target

In [ ]:
# Use the export's own forecast_distance column — DR_RESERVED_PREDICTION_TIMESTAMP is
# just the batch job's run time, not the prediction week, so it can't be used to derive forecast_step.
if TS_FD_COL in ts_target.columns:
    # Multiple job runs can cover the same (START_OF_WEEK, forecast_distance) — keep the most recent.
    ts_target_dedup = (
        ts_target.sort_values(TS_POINT_COL)
        .drop_duplicates(subset=[TS_FD_COL], keep="last")
    )

    ts_export_a = (
        ts_target_dedup[[TS_POINT_COL, TS_FD_COL, TS_PRED_COL]]
        .rename(columns={
            TS_POINT_COL: "prediction_week_dt",
            TS_FD_COL: "forecast_step",
            TS_PRED_COL: f"{pred_col}_EXPORT_A",
        })
        .sort_values("forecast_step")
        .reset_index(drop=True)
    )
    ts_export_a["prediction_week"] = (
        pd.Timestamp(TARGET_WEEK) - pd.to_timedelta(ts_export_a["forecast_step"] * 7, unit="D")
    ).dt.strftime("%Y-%m-%d")
    display(ts_export_a)
else:
    print(f"Column '{TS_FD_COL}' not found. Available:", list(ts_target.columns))
    ts_export_a = None

### 4b — `ts_historical_prediction_data.csv` (130 rows, 43 cols)

Full historical batch-prediction data with all features — **one row per historical target
week**, each carrying a negative `FORECAST_DISTANCE` (weeks before the export's snapshot
date). This is a backtest/monitoring export, not a forward FD1...FD13 fan-out, so it isn't
comparable to Cache/Live/predictions-export on `forecast_step`. It's used below only to
inspect the **feature inputs** DR's batch job actually scored against for this week.

In [ ]:
print(f"Loading ts_historical_prediction_data from catalog: {DR_HISTORICAL_PRED_DATASET_ID}")
ts_hist = dr.Dataset.get(DR_HISTORICAL_PRED_DATASET_ID).get_as_dataframe()
print("Shape:", ts_hist.shape)
print("Columns:", list(ts_hist.columns))
ts_hist.head(3)

In [ ]:
ts_hist.dtypes

In [ ]:
HIST_TARGET_COL = "START_OF_WEEK"   # target date
HIST_POINT_COL  = "DR_RESERVED_PREDICTION_TIMESTAMP" if "DR_RESERVED_PREDICTION_TIMESTAMP" in ts_hist.columns else "FORECAST_POINT"
HIST_PRED_COL   = target_col if target_col in ts_hist.columns else pred_col  # export stores predictions under the target's own column name
HIST_FD_COL     = "FORECAST_DISTANCE"  # backtest offset (weeks before snapshot date) — informational only, not a forecast_step

hist_target = ts_hist[ts_hist[HIST_TARGET_COL].astype(str).str[:10] == TARGET_WEEK].copy()

# A target week can appear in more than one batch run — keep the most recent.
hist_target = hist_target.sort_values(HIST_POINT_COL).drop_duplicates(subset=[HIST_TARGET_COL], keep="last")

print(f"Rows for {TARGET_WEEK}: {len(hist_target)}")
print(f"HIST_POINT_COL: {HIST_POINT_COL} | HIST_PRED_COL: {HIST_PRED_COL} | HIST_FD_COL: {HIST_FD_COL}")

# Plain-English translation of FORECAST_DISTANCE: it's negative because this row was fed into
# the batch job as *known history*, not as a forecasted row. FD = -N means TARGET_WEEK is N
# periods before that job's own forecast reference point (anchor_week below).
if HIST_FD_COL in hist_target.columns and not hist_target.empty:
    for _, _row in hist_target.iterrows():
        fd = _row[HIST_FD_COL]
        anchor_week = (pd.Timestamp(TARGET_WEEK) - pd.to_timedelta(fd * 7, unit="D")).strftime("%Y-%m-%d")
        run_time = _row[HIST_POINT_COL]
        if fd < 0:
            role = f"fed in as known history, {abs(fd)} week(s) before that job's forecast reference point ({anchor_week})"
        elif fd > 0:
            role = f"one of the rows actually being forecasted, {fd} week(s) ahead of that job's reference point ({anchor_week})"
        else:
            role = f"exactly at that job's forecast reference point ({anchor_week})"
        print(f"  FORECAST_DISTANCE={fd} → {TARGET_WEEK} was {role}. Batch job ran at {run_time}.")

hist_target

## 5 — Value comparison: Cache / Live / DR Predictions Export

All three sources share the same weeks-ahead `forecast_step` semantics, so they can be
joined directly. `ts_historical_prediction_data.csv` is excluded here — see §4b/§6.

In [ ]:
# Merge planned scenario only (DR export has no scenario split)
planned_comparison = comparison[comparison["scenario"] == "planned"].copy()
planned_comparison = planned_comparison.rename(columns={
    "diff": "diff_live_vs_cache",
    "diff_%": "diff_live_vs_cache_%",
})

if ts_export_a is not None:
    planned_comparison = planned_comparison.merge(
        ts_export_a[["prediction_week", "forecast_step", f"{pred_col}_EXPORT_A"]],
        on=["prediction_week", "forecast_step"],
        how="outer",
    )
    export_col = f"{pred_col}_EXPORT_A"
    planned_comparison["diff_live_vs_export"] = (
        planned_comparison[f"{pred_col}_LIVE"] - planned_comparison[export_col]
    ).round(3)
    planned_comparison["diff_live_vs_export_%"] = (
        planned_comparison["diff_live_vs_export"] / planned_comparison[export_col] * 100
    ).round(2)
    planned_comparison["diff_cache_vs_export"] = (
        planned_comparison[f"{pred_col}_CACHE"] - planned_comparison[export_col]
    ).round(3)
    planned_comparison["diff_cache_vs_export_%"] = (
        planned_comparison["diff_cache_vs_export"] / planned_comparison[export_col] * 100
    ).round(2)

planned_comparison.sort_values("forecast_step").reset_index(drop=True)

## 6 — Feature-input diff: DR's historical batch job vs our reconstructed scoring input

`ts_historical_prediction_data.csv` isn't comparable on predicted value (§4b) — for a
negative-`FORECAST_DISTANCE` row, `HIST_PRED_COL` holds the **actual observed value**, not a
model prediction (that row was fed in as known history, not forecasted). What it *does* give
us is the exact feature values DR's own batch job scored against for `TARGET_WEEK`.

Live/cache calls build their scoring input from the `PLANNED`/`ACTUAL` feature datasets at
whatever prediction week we choose — if those datasets have since been updated (backfilled
actuals, corrected plans, etc.), the inputs diverge from what DR's batch job saw. This is a
**data-lineage check**, not a forecast-accuracy check.

§6.1/§6.2 below show each source's raw feature row; §6.3 aligns them into a single diff table
so mismatches are immediately visible — pick `INSPECT_PRED_WEEK`/`INSPECT_SCENARIO` in §6.2
to control which reconstructed input is compared against.

In [ ]:
# 6.1 — Feature values DR's historical batch job scored against for TARGET_WEEK
# (FORECAST_DISTANCE kept as informational context — it's a backtest offset, not a forecast_step.
#  HIST_PRED_COL here is the ACTUAL observed value, not a prediction — see markdown above.)
skip_cols = {HIST_TARGET_COL, HIST_POINT_COL, HIST_PRED_COL, "ASSOCIATION_ID"}
feature_cols_export = [c for c in hist_target.columns if c not in skip_cols]
print(f"{len(feature_cols_export)} feature columns in DR export")

hist_target[feature_cols_export]

In [ ]:
# 6.2 — Feature values from our reconstructed scoring input (FD1 example)
INSPECT_PRED_WEEK = "2026-05-04"  # FD1
INSPECT_SCENARIO  = "planned"

fw_src = planned_src if INSPECT_SCENARIO == "planned" else actual_src
scoring_input = _build_scoring_input_for_week(
    scoring_df=scoring_src,
    prediction_week=INSPECT_PRED_WEEK,
    fw_features_df=fw_src,
)
fw_rows = scoring_input[scoring_input[target_col].isna()]
fw_target = fw_rows[fw_rows[date_col].astype(str).str[:10] == TARGET_WEEK]

feature_cols_local = [c for c in fw_target.columns if c not in [
    date_col, target_col, "ASSOCIATION_ID", "SKILL"
]]
print(f"FW row for {TARGET_WEEK} at prediction_week={INSPECT_PRED_WEEK}:")
fw_target[feature_cols_local].T.rename(columns={fw_target.index[0]: "value"})

In [ ]:
# 6.3 — Side-by-side feature diff: DR export vs reconstructed input
common_features = [c for c in feature_cols_export if c in feature_cols_local]
missing_from_export  = [c for c in feature_cols_local  if c not in feature_cols_export]
missing_from_recon   = [c for c in feature_cols_export if c not in feature_cols_local]

export_vals = hist_target[common_features].iloc[0]
recon_vals  = fw_target[common_features].iloc[0]

feature_diff = pd.DataFrame({
    "DR_export_value": export_vals,
    "reconstructed_value": recon_vals,
})
feature_diff["numeric_diff"] = (
    pd.to_numeric(feature_diff["DR_export_value"], errors="coerce")
    - pd.to_numeric(feature_diff["reconstructed_value"], errors="coerce")
)
feature_diff["match"] = feature_diff["DR_export_value"].astype(str) == feature_diff["reconstructed_value"].astype(str)

mismatches = feature_diff[~feature_diff["match"]]
print(f"Comparing DR export inputs for {TARGET_WEEK} vs reconstructed '{INSPECT_SCENARIO}' input "
      f"at prediction_week={INSPECT_PRED_WEEK}")
print(f"{len(mismatches)} / {len(feature_diff)} common feature columns differ")
if missing_from_export:
    print(f"Columns only in reconstructed input (not in DR export): {missing_from_export}")
if missing_from_recon:
    print(f"Columns only in DR export (not in reconstructed input): {missing_from_recon}")

# Mismatches first
feature_diff.sort_values("match")